### Harmonic Oscillator
$m\frac{d^2y(t)}{dt^2}+\mu\frac{dy(t)}{dt}+ k y(t)=0$

$m$: mass ($m=1$)

$k$: spring constant ($k=101$)

$\mu$: friction coefficient ($\mu=2$)

$y(0)=1$

In [ ]:
import torch
import torch.nn as nn
import mlp
import utils
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
# actual solution
m = 1
mu = 2
k = 101
d = mu/(2*m)
w = np.sqrt(k/m-d**2) ** 0.5
t = torch.linspace(0, 1, 50).to(device).reshape(-1, 1)
y_sol = torch.exp(-d*t) * torch.cos(w*t)
y_ini = y_sol[0].item()
print(f"w = {w:.4f}, y(0) = {y_ini:.4f}")

In [ ]:
num_hidden = 4
num_nodes = 32
layer_list = [1] + [num_nodes] * num_hidden + [1]
model = mlp.MLP(layer_list).to(device)

lr = 2e-3
num_epochs = 20000
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

In [ ]:
ls = 10

t_req = t.clone()
t_req.requires_grad = True

for _ in range(num_epochs):
    optimizer.zero_grad()
    y0_pred = model(t[0:1])
    y_pred = model(t_req)
    loss_col = torch.mean(utils.burgers_equation(y_pred, t_req)**2)
    loss_ini = torch.mean((y_pred[0:1] - y_ini) ** 2)
    loss = loss_col + ls * loss_ini
    
    loss.backward()
    optimizer.step()
    
    if loss.item() < ls:
        ls = loss.item()
        torch.save(model.state_dict(), './param/ocs.pt')
        
print(f'mse: {ls}')
print(f'y0_pred: {y0_pred.item()}')

In [ ]:
# result
t = torch.linspace(0,1,100).to(device).reshape(-1,1)
ysol = torch.exp(-t)*torch.cos(w*t)

model.load_state_dict(torch.load('./params/ocs.pt',map_location=device))
with torch.no_grad():
    y_pred = model(t)
    
plt.figure(figsize=(8, 4))
plt.plot(t.cpu().numpy(),y_sol.cpu().numpy(),'--',color='black')
plt.plot(t.cpu().numpy(),y_pred.detach().cpu().numpy(),color='red',linewidth=1)
plt.legend(['solution','PINN'])
plt.ylabel('displacement y')
plt.xlabel('time t')
plt.title(rf'$m=1$, $\mu={mu}$, $k={k}$, $\omega={w}$, $y_0={yini}$')
plt.show()